# EVOLVE – Model Training (YOLOv8s)

This notebook trains and adapts a YOLOv8s object detection model on the EVOLVE dataset.

The objective is to evaluate the impact of domain adaptation under extreme low-light and high-density concert environments.

This notebook performs:

1. Dataset verification
2. Baseline evaluation (COCO pre-trained model)
3. Fine-tuning on the EVOLVE training set
4. Validation-based performance comparison
5. Export of best model and training metadata

The test set is strictly excluded from this notebook.

## Experimental Setup

In [ ]:
# ==============================
# Imports
# ==============================

import torch
from ultralytics import YOLO
import yaml
import os
import json
import pandas as pd
from pathlib import Path
import shutil

# ==============================
# Reproducibility
# ==============================

SEED = 42
torch.manual_seed(SEED)

In [ ]:
# ==============================
# Device configuration
# ==============================

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = 0
else:
    device = "cpu"

print("Device used:", device)
print("PyTorch version:", torch.__version__)

In [ ]:
# ==============================
# Dataset configuration
# ==============================

DATASET_CONFIG_PATH = "../configs/dataset.yaml"

with open(DATASET_CONFIG_PATH, "r") as f:
    dataset_config = yaml.safe_load(f)

print("Train path:", dataset_config.get("train"))
print("Val path:", dataset_config.get("val"))

class_names = dataset_config["names"]

if isinstance(class_names, dict):
    class_names = list(class_names.values())

print("Number of classes:", len(class_names))
print("Class names:", class_names)

In [ ]:
BASE_PATH = Path(dataset_config["path"]).resolve()

train_path = BASE_PATH / dataset_config["train"]
val_path = BASE_PATH / dataset_config["val"]

print("Resolved train path:", train_path)
print("Resolved val path:", val_path)

In [ ]:
# ==============================
# Dataset volume verification
# ==============================

train_images = list(train_path.glob("*.*"))
val_images = list(val_path.glob("*.*"))

print("Number of training images:", len(train_images))
print("Number of validation images:", len(val_images))

In [ ]:
# ==============================
# Class distribution analysis
# ==============================

from collections import defaultdict

class_counts = defaultdict(int)

label_train_path = BASE_PATH / "labels" / "train"

label_files = list(label_train_path.glob("*.txt"))

for label_file in label_files:
    with open(label_file, "r") as f:
        lines = f.readlines()
        for line in lines:
            class_id = int(line.split()[0])
            class_counts[class_id] += 1

# Convert to readable format
class_names = dataset_config["names"]

distribution = {
    class_names[class_id]: count
    for class_id, count in class_counts.items()
}

print("Class distribution (training set):")
for k, v in distribution.items():
    print(f"{k}: {v}")

In [ ]:
# ==============================
# Class distribution visualization
# ==============================

import matplotlib.pyplot as plt

classes = list(distribution.keys())
counts = list(distribution.values())

plt.figure()
plt.bar(classes, counts)
plt.xticks(rotation=45)
plt.xlabel("Classes")
plt.ylabel("Number of instances")
plt.title("Training Set Class Distribution")
plt.tight_layout()
plt.show()

## Baseline Evaluation (YOLOv8s COCO)

In [ ]:
# ==============================
# Baseline evaluation
# ==============================

baseline_model = YOLO("yolov8s.pt")

baseline_metrics = baseline_model.val(
    data=DATASET_CONFIG_PATH,
    split="val",
    device=device
)

print("\nBaseline Validation Results:")
print("mAP50-95:", baseline_metrics.box.map)
print("mAP50:", baseline_metrics.box.map50)
print("Precision:", baseline_metrics.box.mp)
print("Recall:", baseline_metrics.box.mr)

## Fine-Tuning YOLOv8s

In [ ]:
# ==============================
# Fine-tuning with Early Stopping
# ==============================

model = YOLO("yolov8s.pt")

results = model.train(
    data=DATASET_CONFIG_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    seed=SEED,
    patience=20,  # Early stopping
    project="../runs/train",
    name="evolve_yolov8s",
    exist_ok=True
)

In [ ]:
# ==============================
# Load training logs
# ==============================

import pandas as pd

results_csv_path = results.save_dir / "results.csv"

training_log = pd.read_csv(results_csv_path)

print("Training log loaded.")
training_log.head()

In [ ]:
# ==============================
# Plot Loss Curves
# ==============================

import matplotlib.pyplot as plt

plt.figure()

plt.plot(training_log["epoch"], training_log["train/box_loss"], label="Train Box Loss")
plt.plot(training_log["epoch"], training_log["val/box_loss"], label="Val Box Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Box Loss")
plt.legend()
plt.show()

In [ ]:
# ==============================
# Plot mAP Curve
# ==============================

plt.figure()

plt.plot(training_log["epoch"], training_log["metrics/mAP50"], label="mAP@0.5")
plt.plot(training_log["epoch"], training_log["metrics/mAP50-95"], label="mAP@0.5:0.95")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation mAP Evolution")
plt.legend()
plt.show()

In [ ]:
# ==============================
# Save training figures
# ==============================

os.makedirs("../results/figures", exist_ok=True)

plt.figure()
plt.plot(training_log["epoch"], training_log["train/box_loss"])
plt.plot(training_log["epoch"], training_log["val/box_loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Box Loss Evolution")
plt.savefig("../results/figures/loss_curve.png")
plt.close()

plt.figure()
plt.plot(training_log["epoch"], training_log["metrics/mAP50"])
plt.plot(training_log["epoch"], training_log["metrics/mAP50-95"])
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("mAP Evolution")
plt.savefig("../results/figures/map_curve.png")
plt.close()

print("Training figures saved.")

## Validation After Fine-Tuning

In [ ]:
# ==============================
# Validation after fine-tuning
# ==============================

best_model_path = results.save_dir / "weights" / "best.pt"
finetuned_model = YOLO(best_model_path)

finetuned_metrics = finetuned_model.val(
    data=DATASET_CONFIG_PATH,
    split="val",
    device=device
)

print("\nFine-Tuned Validation Results:")
print("mAP50-95:", finetuned_metrics.box.map)
print("mAP50:", finetuned_metrics.box.map50)
print("Precision:", finetuned_metrics.box.mp)
print("Recall:", finetuned_metrics.box.mr)

## Baseline vs Fine-Tuned Comparison

In [ ]:
# ==============================
# Comparison table
# ==============================

comparison = pd.DataFrame({
    "Metric": ["mAP50-95", "mAP50", "Precision", "Recall"],
    "Baseline": [
        baseline_metrics.box.map,
        baseline_metrics.box.map50,
        baseline_metrics.box.mp,
        baseline_metrics.box.mr
    ],
    "Fine-Tuned": [
        finetuned_metrics.box.map,
        finetuned_metrics.box.map50,
        finetuned_metrics.box.mp,
        finetuned_metrics.box.mr
    ]
})

comparison

## Exports

In [ ]:
# ==============================
# Export metrics
# ==============================

os.makedirs("../results", exist_ok=True)

baseline_dict = {
    "mAP50-95": float(baseline_metrics.box.map),
    "mAP50": float(baseline_metrics.box.map50),
    "Precision": float(baseline_metrics.box.mp),
    "Recall": float(baseline_metrics.box.mr)
}

finetuned_dict = {
    "mAP50-95": float(finetuned_metrics.box.map),
    "mAP50": float(finetuned_metrics.box.map50),
    "Precision": float(finetuned_metrics.box.mp),
    "Recall": float(finetuned_metrics.box.mr)
}

with open("../results/baseline_metrics.json", "w") as f:
    json.dump(baseline_dict, f, indent=4)

with open("../results/finetuned_metrics.json", "w") as f:
    json.dump(finetuned_dict, f, indent=4)

comparison.to_csv("../results/training_summary.csv", index=False)

print("Metrics exported successfully.")

In [ ]:
# ==============================
# Export best model
# ==============================

shutil.copy(best_model_path, "../results/best_model.pt")

print("Best model exported from:", best_model_path)

# ==============================
# Export training metadata
# ==============================

training_metadata = {
    "model_architecture": "yolov8s",
    "epochs": 50,
    "imgsz": 640,
    "batch_size": 16,
    "seed": SEED,
    "train_images": len(train_images),
    "val_images": len(val_images)
}

with open("../results/training_config.json", "w") as f:
    json.dump(training_metadata, f, indent=4)

print("Training configuration exported.")

## Conclusion

This notebook demonstrated:

- Baseline performance of a pre-trained YOLOv8s detector
- Performance after domain-specific fine-tuning
- Controlled validation comparison

The selected model will now be evaluated on the held-out test set in Notebook 2.

## Conclusion

This notebook demonstrated:

- Baseline performance of a pre-trained detector
- Performance after domain-specific fine-tuning
- Controlled comparison on validation data

The next step consists of evaluating the selected model on the held-out test set.